In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data = {
    'glucose':        [85, 183, 89, 137, 116, 78, 115, 197, 125, 110,
                       168, 139, 189, 166, 100, 118, 107, 103, 115, 126,
                       99, 196, 119, 143, 125, 147, 72, 500, 93, 88],
    'blood_pressure': [66, 64, 66, 40, 74, 50, 0, 70, 96, 92,
                       74, 80, 60, 72, 0, 84, 80, 30, 70, 88,
                       60, 70, 0, 80, 70, 110, 60, 0, 100, 58],
    'bmi':            [26.6, 23.3, 28.1, 43.1, 25.6, 31.0, 35.3, 30.5, 0.0, 37.6,
                       38.2, 27.1, 30.1, 25.8, 30.0, 27.3, 30.5, 27.5, 29.0, 30.1,
                       25.0, 25.0, 29.6, 33.2, 32.0, 28.1, 36.9, 0.0, 38.0, 26.0],
    'age':            [31, 32, 21, 33, 30, 26, 29, 53, 54, 31,
                       35, 57, 59, 51, 32, 33, 40, 27, 23, 29,
                       28, 41, 25, 52, 31, 37, 22, 90, 36, 29],
    'diabetes':       [0, 1, 0, 1, 0, 1, 0, 1, 1, 0,
                       1, 0, 1, 1, 1, 0, 0, 0, 0, 1,
                       0, 0, 0, 1, 0, 1, 0, 1, 0, 0]
}

df = pd.DataFrame(data)


In [ ]:
from numpy.strings import upper
from numpy._core.defchararray import lower
# -------------------------------
# Task 1 — Outlier Detection & Capping (IQR)
# -------------------------------


numerical_cols = ['glucose', 'bmi', 'age', 'blood_pressure']

print('outline capping summary:')

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q2 = df[col].quantile(0.50)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q3

    lower_whisker = Q1 - 1.5 * IQR
    upper_whisker = Q3 + 1.5 * IQR

    # count outline before capping
    lower_outline = (df[col] < lower_whisker).sum()
    upper_outline = (df[col] > upper_whisker).sum()
    total_count = lower_outline + upper_outline

    # Cap values
    df[col] = np.where(df[col] < lower_whisker, lower_whisker, df[col])
    df[col] = np.where(df[col] > upper_whisker, upper_whisker, df[col])

    print(f"{col}: {total_count} value capped")



outline capping summary:
glucose: 16 value capped
bmi: 16 value capped
age: 15 value capped
blood_pressure: 14 value capped


In [ ]:
# -------------------------------
# Task 2 — Split & Scale
# -------------------------------

X = df.drop(columns=["diabetes"])
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split (
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)



In [ ]:
from numpy.random.mtrand import random
# -------------------------------
# Task 3 — Train & Evaluate
# -------------------------------


model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Accuracy

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)


print("\nModel Performance:")
print(f"Traing Accuracy: {train_accuracy:.4f}")
print(f"test Accuracy: {test_accuracy:.4f}")

# COMMENT:
# If train accuracy is much higher than test → Overfitting
# If both are low → Underfitting
# If both are similar and reasonably high → Healthy Generalisation
# (In this case, accuracies are usually close → suggests healthy generalisation)



Model Performance:
Traing Accuracy: 0.7143
test Accuracy: 1.0000


In [ ]:
# -------------------------------
# Task 4 — Feature Correlation
# -------------------------------

print("/nCorrelation Matrix:")
corr_matrix = df.corr()
print(corr_matrix)

# Find highest absolute correlation pair (excluding self-correlation)
corr_matrix_abs = corr_matrix.abs()

# Set diagonal to 0 to ignore self-correlation
np.fill_diagonal(corr_matrix_abs.values, 0)

max_corr = corr_matrix_abs.max().max()

# Get feature pair
max_pair = np.where(corr_matrix_abs == max_corr)
feature_1 = corr_matrix_abs.index[max_pair[0][0]]
feature_2 = corr_matrix_abs.columns[max_pair[1][0]]

print(f"\nHighest Correlation Pair: {feature_1} & {feature_2} = {max_corr:.4f}")


# COMMENT:
# Correlation > 0.8 → Strong multicollinearity concern
# Correlation between 0.5–0.8 → Moderate concern
# Correlation < 0.5 → Usually safe
# (Based on result, decide if multicollinearity is an issue)


/nCorrelation Matrix:
                 glucose  blood_pressure       bmi       age  diabetes
glucose         1.000000       -0.277506 -0.558864  0.820174  0.411558
blood_pressure -0.277506        1.000000  0.086589 -0.037836 -0.008585
bmi            -0.558864        0.086589  1.000000 -0.576059 -0.190735
age             0.820174       -0.037836 -0.576059  1.000000  0.461545
diabetes        0.411558       -0.008585 -0.190735  0.461545  1.000000

Highest Correlation Pair: glucose & age = 0.8202
